# Embeddings From Scratch (BPE + SGNS + LLM Ready)

This notebook builds high-quality token embeddings from your trained byte-level BPE tokenizer.

It includes:
- loading tokenizer from JSON
- encoding corpus to token IDs
- training skip-gram with negative sampling (SGNS)
- exporting learned embedding weights for the attention notebook
- hardware-aware tuning profiles for CPU and RTX-class GPUs

In [1]:
from __future__ import annotations

import json
import random
import re
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")

Device: cpu


In [2]:
@dataclass
class TokenizerConfig:
    vocab_size: int
    min_pair_freq: int
    special_tokens: Tuple[str, ...]


class BPETokenizerRuntime:
    """Runtime byte-level BPE tokenizer loader for embedding training."""

    _word_re = re.compile(r"\s+|[^\s]+")

    def __init__(self, config: TokenizerConfig, merges_raw: List[List[int]]):
        self.config = config
        self.base_vocab_size = 256
        self.special_tokens = list(config.special_tokens)

        self.special_to_id: Dict[str, int] = {}
        self.id_to_special: Dict[int, str] = {}
        self.merges: Dict[Tuple[int, int], int] = {}
        self.merges_rank: Dict[Tuple[int, int], int] = {}

        self.token_to_bytes: Dict[int, bytes] = {
            i: bytes([i]) for i in range(self.base_vocab_size)
        }
        self._init_special_tokens()

        for rank, (a, b, new_id) in enumerate(merges_raw):
            pair = (int(a), int(b))
            merged_id = int(new_id)
            self.merges[pair] = merged_id
            self.merges_rank[pair] = rank
            self.token_to_bytes[merged_id] = (
                self.token_to_bytes[pair[0]] + self.token_to_bytes[pair[1]]
            )

    @classmethod
    def load(cls, path: str | Path) -> "BPETokenizerRuntime":
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        conf = payload["config"]
        config = TokenizerConfig(
            vocab_size=int(conf["vocab_size"]),
            min_pair_freq=int(conf.get("min_pair_freq", 2)),
            special_tokens=tuple(conf.get("special_tokens", ["<pad>", "<bos>", "<eos>", "<unk>"])),
        )
        return cls(config, payload["merges"])

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_bytes)

    def _init_special_tokens(self) -> None:
        start = self.base_vocab_size
        for i, tok in enumerate(self.special_tokens):
            tid = start + i
            self.special_to_id[tok] = tid
            self.id_to_special[tid] = tok
            self.token_to_bytes[tid] = tok.encode("utf-8")

    def _encode_chunk(self, chunk: str) -> List[int]:
        symbols: List[int] = list(chunk.encode("utf-8"))
        if len(symbols) < 2:
            return symbols

        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")

            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                rank = self.merges_rank.get(pair)
                if rank is not None and rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            merged_token = self.merges[best_pair]
            out: List[int] = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                    out.append(merged_token)
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            symbols = out

        return symbols

    def encode(self, text: str, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        if text == "":
            return []

        token_ids: List[int] = []
        if add_bos and "<bos>" in self.special_to_id:
            token_ids.append(self.special_to_id["<bos>"])

        for chunk in self._word_re.findall(text):
            token_ids.extend(self._encode_chunk(chunk))

        if add_eos and "<eos>" in self.special_to_id:
            token_ids.append(self.special_to_id["<eos>"])

        return token_ids

    def token_text(self, token_id: int) -> str:
        if token_id in self.id_to_special:
            return self.id_to_special[token_id]
        return self.token_to_bytes[token_id].decode("utf-8", errors="replace")

In [3]:
project_root = Path("..")
tokenizer_path = Path("bpe_tokenizer_wizard.json")
data_path = project_root / "wizard_of_oz.txt"

if not tokenizer_path.exists():
    raise FileNotFoundError(
        "Tokenizer JSON not found at Research/bpe_tokenizer_wizard.json. "
        "Run the tokenizer notebook first."
    )

if not data_path.exists():
    raise FileNotFoundError("wizard_of_oz.txt not found in project root.")

tokenizer = BPETokenizerRuntime.load(tokenizer_path)
text = data_path.read_text(encoding="utf-8")
token_ids = tokenizer.encode(text, add_bos=True, add_eos=True)
token_freq = Counter(token_ids)
vocab_size = tokenizer.vocab_size

print(f"Corpus length (chars): {len(text):,}")
print(f"Token count: {len(token_ids):,}")
print(f"Tokenizer vocab size: {vocab_size:,}")
print("Top 10 token IDs by frequency:", token_freq.most_common(10))

Corpus length (chars): 232,309
Token count: 102,130
Tokenizer vocab size: 2,000
Top 10 token IDs by frequency: [(32, 38298), (261, 2900), (10, 2496), (268, 1491), (270, 1342), (269, 1126), (44, 1019), (97, 988), (282, 966), (46, 814)]


In [4]:
@dataclass
class EmbeddingTrainingConfig:
    dim: int
    window_size: int
    negatives: int
    batch_size: int
    epochs: int
    lr: float
    max_pairs: int
    grad_accum_steps: int = 1


def estimate_sgns_memory_gb(vocab_size: int, dim: int, mixed_precision: bool) -> float:
    # SGNS has two embedding tables: input and output.
    param_count = 2 * vocab_size * dim
    bytes_per_param = 10 if mixed_precision else 16
    return (param_count * bytes_per_param) / (1024 ** 3)


def build_profiles() -> Dict[str, EmbeddingTrainingConfig]:
    return {
        "cpu_safe": EmbeddingTrainingConfig(
            dim=128, window_size=4, negatives=5, batch_size=256, epochs=2, lr=2e-3, max_pairs=220_000
        ),
        "cpu_quality": EmbeddingTrainingConfig(
            dim=192, window_size=5, negatives=6, batch_size=256, epochs=3, lr=1.5e-3, max_pairs=320_000
        ),
        "rtx_4060_balanced": EmbeddingTrainingConfig(
            dim=256, window_size=5, negatives=8, batch_size=1024, epochs=3, lr=2e-3, max_pairs=900_000
        ),
        "rtx_4060_quality": EmbeddingTrainingConfig(
            dim=384, window_size=6, negatives=10, batch_size=1024, epochs=4, lr=1.5e-3, max_pairs=1_400_000
        ),
        "rtx_4060_max": EmbeddingTrainingConfig(
            dim=512, window_size=6, negatives=12, batch_size=768, epochs=4, lr=1.2e-3, max_pairs=1_800_000, grad_accum_steps=2
        ),
    }


profiles = build_profiles()
gpu_vram_gb = 0.0
if device.type == "cuda":
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

if device.type == "cuda" and gpu_vram_gb >= 7.5:
    selected_profile = "rtx_4060_quality"
elif device.type == "cuda":
    selected_profile = "rtx_4060_balanced"
else:
    selected_profile = "cpu_safe"

cfg = profiles[selected_profile]
use_mixed_precision = device.type == "cuda"

print("Selected profile:", selected_profile)
print("Config:", cfg)
for name, p in profiles.items():
    mem_gb = estimate_sgns_memory_gb(vocab_size, p.dim, mixed_precision=(device.type == "cuda"))
    print(f"{name:18s} | dim={p.dim:4d} | est_sgns_mem={mem_gb:.3f} GB")

Selected profile: cpu_safe
Config: EmbeddingTrainingConfig(dim=128, window_size=4, negatives=5, batch_size=256, epochs=2, lr=0.002, max_pairs=220000, grad_accum_steps=1)
cpu_safe           | dim= 128 | est_sgns_mem=0.008 GB
cpu_quality        | dim= 192 | est_sgns_mem=0.011 GB
rtx_4060_balanced  | dim= 256 | est_sgns_mem=0.015 GB
rtx_4060_quality   | dim= 384 | est_sgns_mem=0.023 GB
rtx_4060_max       | dim= 512 | est_sgns_mem=0.031 GB


In [5]:
def build_skipgram_pairs(
    ids: List[int],
    window_size: int,
    max_pairs: int,
    seed: int = 42,
    skip_token_ids: set[int] | None = None,
    ) -> List[Tuple[int, int]]:
    rng = random.Random(seed)
    skip_token_ids = skip_token_ids or set()

    pairs: List[Tuple[int, int]] = []
    n = len(ids)

    for i, center in enumerate(ids):
        if center in skip_token_ids:
            continue

        w = rng.randint(1, window_size)
        left = max(0, i - w)
        right = min(n, i + w + 1)

        contexts = [ids[j] for j in range(left, right) if j != i and ids[j] not in skip_token_ids]
        if not contexts:
            continue

        context = rng.choice(contexts)
        pairs.append((center, context))

        if len(pairs) >= max_pairs:
            break

    return pairs


class SkipGramPairsDataset(Dataset):
    def __init__(self, pairs: List[Tuple[int, int]]):
        self.centers = torch.tensor([c for c, _ in pairs], dtype=torch.long)
        self.contexts = torch.tensor([ctx for _, ctx in pairs], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.centers)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.centers[idx], self.contexts[idx]


def build_noise_distribution(ids: List[int], vocab_size: int, power: float = 0.75) -> torch.Tensor:
    freqs = np.zeros(vocab_size, dtype=np.float64)
    for tid, count in Counter(ids).items():
        if 0 <= tid < vocab_size:
            freqs[tid] = count

    freqs = np.maximum(freqs, 1e-12)
    probs = freqs ** power
    probs /= probs.sum()
    return torch.tensor(probs, dtype=torch.float32)


class SkipGramNS(nn.Module):
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        self.input_embeddings = nn.Embedding(vocab_size, dim)
        self.output_embeddings = nn.Embedding(vocab_size, dim)

        init_range = 0.5 / dim
        nn.init.uniform_(self.input_embeddings.weight, -init_range, init_range)
        nn.init.zeros_(self.output_embeddings.weight)

    def forward(
        self,
        centers: torch.Tensor,
        positive_contexts: torch.Tensor,
        negative_contexts: torch.Tensor,
    ) -> torch.Tensor:
        center_vecs = self.input_embeddings(centers)
        pos_vecs = self.output_embeddings(positive_contexts)
        neg_vecs = self.output_embeddings(negative_contexts)

        pos_scores = torch.sum(center_vecs * pos_vecs, dim=1)
        pos_loss = F.logsigmoid(pos_scores)

        neg_scores = torch.bmm(neg_vecs, center_vecs.unsqueeze(2)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_scores).sum(dim=1)

        loss = -(pos_loss + neg_loss).mean()
        return loss

In [9]:
def train_skipgram(
    model: SkipGramNS,
    loader: DataLoader,
    noise_dist: torch.Tensor,
    cfg: EmbeddingTrainingConfig,
    device: torch.device,
    ) -> List[float]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    model.train()
    noise_dist = noise_dist.to(device)
    history: List[float] = []

    for epoch in range(cfg.epochs):
        running_loss = 0.0
        progress = tqdm(loader, desc=f"Epoch {epoch + 1}/{cfg.epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for step, (centers, pos_contexts) in enumerate(progress, start=1):
            centers = centers.to(device)
            pos_contexts = pos_contexts.to(device)

            neg_contexts = torch.multinomial(
                noise_dist,
                centers.size(0) * cfg.negatives,
                replacement=True,
            ).view(centers.size(0), cfg.negatives)

            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    loss = model(centers, pos_contexts, neg_contexts) / cfg.grad_accum_steps
                scaler.scale(loss).backward()
            else:
                loss = model(centers, pos_contexts, neg_contexts) / cfg.grad_accum_steps
                loss.backward()

            should_step = (step % cfg.grad_accum_steps == 0) or (step == len(loader))
            if should_step:
                if device.type == "cuda":
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            batch_loss = loss.item() * cfg.grad_accum_steps
            running_loss += batch_loss
            progress.set_postfix(loss=f"{batch_loss:.4f}")

        epoch_loss = running_loss / max(len(loader), 1)
        history.append(epoch_loss)
        print(f"Epoch {epoch + 1}/{cfg.epochs} | avg_loss={epoch_loss:.4f}")

    return history


skip_ids = set()
for tok in ("<pad>",):
    if tok in tokenizer.special_to_id:
        skip_ids.add(tokenizer.special_to_id[tok])

pairs = build_skipgram_pairs(
    ids=token_ids,
    window_size=cfg.window_size,
    max_pairs=cfg.max_pairs,
    seed=SEED,
    skip_token_ids=skip_ids,
)

dataset = SkipGramPairsDataset(pairs)
loader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

noise_dist = build_noise_distribution(token_ids, vocab_size=vocab_size)
sgns_model = SkipGramNS(vocab_size=vocab_size, dim=cfg.dim).to(device)

print(f"Training pairs: {len(pairs):,}")
print(f"Batches per epoch: {len(loader):,}")

train_start = time.perf_counter()
loss_history = train_skipgram(sgns_model, loader, noise_dist, cfg, device)
elapsed = time.perf_counter() - train_start
print(f"Training finished in {elapsed:.2f}s")

Training pairs: 102,130
Batches per epoch: 398


Epoch 1/2:   0%|          | 0/398 [00:00<?, ?it/s]

Epoch 1/2 | avg_loss=2.7899


Epoch 2/2:   0%|          | 0/398 [00:00<?, ?it/s]

Epoch 2/2 | avg_loss=2.4509
Training finished in 7.45s


In [10]:
trained_token_embeddings = F.normalize(
    sgns_model.input_embeddings.weight.detach().cpu(), p=2, dim=1
)

def nearest_neighbors(token_id: int, top_k: int = 8) -> List[Tuple[int, float]]:
    query = trained_token_embeddings[token_id]
    sims = trained_token_embeddings @ query
    indices = torch.topk(sims, k=min(top_k + 1, sims.numel())).indices.tolist()

    out: List[Tuple[int, float]] = []
    for idx in indices:
        if idx == token_id:
            continue
        out.append((idx, float(sims[idx])))
        if len(out) >= top_k:
            break
    return out


print("Embedding matrix shape:", tuple(trained_token_embeddings.shape))
for tid, _ in token_freq.most_common(5):
    neigh = nearest_neighbors(tid, top_k=6)
    token_repr = repr(tokenizer.token_text(tid))
    print(f"\nToken {tid} {token_repr}")
    for nid, score in neigh:
        print(f"  -> {nid:4d} {repr(tokenizer.token_text(nid))} | sim={score:.4f}")

Embedding matrix shape: (2000, 128)

Token 32 ' '
  ->   10 '\n' | sim=0.9836
  ->  500 '--' | sim=0.9525
  ->  108 'l' | sim=0.9176
  ->   86 'V' | sim=0.9152
  ->  106 'j' | sim=0.9124
  ->  924 'door' | sim=0.9080

Token 261 'the'
  ->  305 'no' | sim=0.9928
  ->  737 'Land' | sim=0.9917
  ->  269 'to' | sim=0.9912
  ->  600 'made' | sim=0.9911
  ->  454 'when' | sim=0.9908
  ->  282 'of' | sim=0.9905

Token 10 '\n'
  ->   32 ' ' | sim=0.9836
  ->  500 '--' | sim=0.9728
  ->  108 'l' | sim=0.9500
  ->   86 'V' | sim=0.9481
  ->  106 'j' | sim=0.9455
  -> 1223 'ond' | sim=0.9442

Token 268 'and'
  ->  312 'that' | sim=0.9949
  ->  462 'could' | sim=0.9949
  ->  421 'from' | sim=0.9946
  ->  332 'had' | sim=0.9943
  ->  573 'good' | sim=0.9941
  ->  807 'went' | sim=0.9941

Token 270 '\n\n'
  ->  661 '\n\n\n' | sim=0.8088
  -> 1091 '\n\n\n\n\n' | sim=0.7565
  ->  914 '\n\n  ' | sim=0.7247
  ->   83 'S' | sim=0.6966
  -> 1047 'ES' | sim=0.6943
  ->   49 '1' | sim=0.6896


In [11]:
class TokenAndPositionEmbedding(nn.Module):
    """LLM-ready token + learned positional embeddings."""

    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int, token_weight: torch.Tensor | None = None):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)

        if token_weight is not None:
            if token_weight.shape != self.token_embed.weight.shape:
                raise ValueError(
                    f"token_weight shape {tuple(token_weight.shape)} does not match "
                    f"embedding shape {tuple(self.token_embed.weight.shape)}"
                )
            self.token_embed.weight.data.copy_(token_weight)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        # token_ids shape: [batch, seq_len]
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0).expand(batch_size, seq_len)
        return self.token_embed(token_ids) + self.pos_embed(positions)


artifact_path = Path("embedding_sgns_wizard.pt")
torch.save(
    {
        "token_embedding": sgns_model.input_embeddings.weight.detach().cpu(),
        "output_embedding": sgns_model.output_embeddings.weight.detach().cpu(),
        "loss_history": loss_history,
        "profile": selected_profile,
        "config": cfg.__dict__,
        "vocab_size": vocab_size,
        "tokenizer_json": str(tokenizer_path),
    },
    artifact_path,
)
print("Saved embedding artifact:", artifact_path.resolve())

demo_ids = torch.tensor(token_ids[:64], dtype=torch.long).unsqueeze(0)
embed_layer = TokenAndPositionEmbedding(
    vocab_size=vocab_size,
    d_model=cfg.dim,
    max_seq_len=512,
    token_weight=sgns_model.input_embeddings.weight.detach().cpu(),
)
with torch.no_grad():
    demo_out = embed_layer(demo_ids)
print("Token+position output shape:", tuple(demo_out.shape))

Saved embedding artifact: D:\Desktop\Mini_Generative_Pretrained_Transformer\Research\embedding_sgns_wizard.pt
Token+position output shape: (1, 64, 128)


## Embedding Families And Best Use Cases

- SGNS / Word2Vec style: fast to train, strong semantic neighborhoods, great for warm-starting token embeddings.
- Transformer token embeddings (joint LM training): best final quality for autoregressive LLMs once attention stack is ready.
- Positional embeddings (learned): best when max sequence length is known and fixed for your training setup.
- RoPE or sinusoidal positions: preferred for longer-context generalization and architecture portability.
- FastText-style subword embeddings: useful for noisy text and morphology-heavy languages.